# Data Preprocessing 

### starting a trial preprocessing

In [1]:
import pandas as pd
import json
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split



1. load data and interpretation. 


In [2]:
df= pd.read_csv('../dataset/processed/decoded_data.csv')


C:\Users\nitya\AppData\Local\Temp\ipykernel_19008\3522198842.py:1: DtypeWarning: Columns (4,9,10,11,12,13,14,15,16,17,19,20,21,22,23,25,26,27,28,29,44,68,69,85,101,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,205,206,208,209,210,211,212,213,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,253,254,255,258,259,260,266,267,268,288,289,292) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv('../dataset/processed/decoded_data.csv')


In [3]:
df.shape

(433323, 350)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 433323 entries, 0 to 433322
Columns: 350 entries, State_FIPS_Code to Drinking_and_Driving_(Reported_having_driven_at_least_once_when_perhaps_had_too_much_to_drink)
dtypes: float64(38), int64(3), object(309)
memory usage: 1.1+ GB


2. Convert SAS variable names into question based column names and values in as as ber value label

In [5]:

def sas_to_question_columns(selected_columns, codebook):
    """
    Convert SAS variable names into question-based column names.

    Example:
        CVDSTRK3 -> Ever_told_you_had_a_stroke
        _BMI5    -> Body_Mass_Index
    """

    # Create SAS variable -> question lookup
    sas_lookup = {
        variable["sas_variable_name"].strip(): variable.get("question", "").strip()
        for variable in codebook
        if variable.get("sas_variable_name")
    }

    renamed_columns = {}

    for sas_name in selected_columns:

        question = sas_lookup.get(sas_name)

        if question:
            # Replace spaces with underscores
            new_name = "_".join(question.split())

            renamed_columns[sas_name] = new_name

        else:
            # Keep original name if not found
            renamed_columns[sas_name] = sas_name

    return renamed_columns

3. Feature selection

In [6]:
with open("../dataset/Document/codebook.json", "r", encoding="utf-8") as f:
    codebook = json.load(f)

selected_columns = [
    'CVDSTRK3',
    '_AGE80',
    'SEXVAR',
    '_BMI5',
    '_RFHYPE6',
    'DIABETE4',
    'SMOKE100',
    '_SMOKER3',
    '_MICHD',
    'CVDINFR4',
    'CVDCRHD4',
    'TOLDHI3',
    'CHOLMED3',
    'CHCKDNY2',
    'PREDIAB2',
    'EXERANY2',
    '_TOTINDA',
    '_PAINDX3',
    'PAMIN13_',
    '_PA30023',
    'GENHLTH',
    'PHYSHLTH',
    'MENTHLTH',
    'EDUCA',
    'INCOME3',
    'EMPLOY1',
    'MARITAL'
]

rename_map = sas_to_question_columns(
    selected_columns,
    codebook
)


In [7]:

# Keep only columns that actually exist in the dataset
available_columns = [value for key, value in rename_map.items() if value in df.columns]

# Create filtered dataframe
df_selected = df[available_columns].copy()

print("Columns retained:")
print(df_selected.columns.tolist())

print("\nShape:")
print(df_selected.shape)

Columns retained:
['(Ever_told)_(you_had)_a_stroke.', 'Imputed_Age_value_collapsed_above_80', 'Sex_of_Respondent', 'Body_Mass_Index_(BMI)', 'Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional', '(Ever_told)_(you_had)_diabetes?_(If_Yes_-_and_respondent_is_female,_ask_Was_this_only_when_you_were_pregnant?_If_Respondent_says_pre-diabetes_or_borderline_diabetes,_use_response_code_4.)', 'Have_you_smoked_at_least_100_cigarettes_in_your_entire_life?_[Note:_5_packs_=_100_cigarettes]', 'Four-level_smoker_status:_Everyday_smoker,_Someday_smoker,_Former_smoker,_Non-smoker', 'Respondents_that_have_ever_reported_having_coronary_heart_disease_(CHD)_or_myocardial_infarction_(MI)', '(Ever_told)_you_had_a_heart_attack,_also_called_a_myocardial_infarction?', '(Ever_told)_(you_had)_angina_or_coronary_heart_disease?', 'Have_you_ever_been_told_by_a_doctor,_nurse_or_other_health_professional_that_your_cholesterol_is_high?', 'Are_you_currently_taking_medi

4. interpret meaning of values and convert them in nan if they are not uninformative or remove the parts that are just clutter

In [8]:
text_columns = df_selected.select_dtypes(include="object").columns
for col in text_columns:
    df_selected[col] = (
        df_selected[col]
        .str.split(" - ", n=1).str[0]
        .str.split(" Notes", n=1).str[0]
        .str.strip()
        .replace({
            "Don't know/Not sure": np.nan,
            "Refused": np.nan,
            "Don't know/Refused/Missing Notes: SMOKE100 = 1 and SMOKEDAY = 9 or SMOKE100 = 7 or 9 or Missing": np.nan,
            "Don't know/Refused/Missing": np.nan,
            "Don't know/Not Sure/Refused/Missing": np.nan,
            "Don't know/Not Sure": np.nan
            
        })
    )


5. Remove row that contains nan value for target variable for cleaning 

In [9]:
df_selected.dropna(subset=["(Ever_told)_(you_had)_a_stroke."], inplace=True)

In [10]:
text_columns = df_selected.select_dtypes(include="object").columns
for col in text_columns:
    print(f"Column: {col}")
    print(df_selected[col].unique())
    print("\n")

Column: (Ever_told)_(you_had)_a_stroke.
['No' 'Yes']


Column: Sex_of_Respondent
['Female' 'Male']


Column: Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional
['Yes' 'No' nan]


Column: (Ever_told)_(you_had)_diabetes?_(If_Yes_-_and_respondent_is_female,_ask_Was_this_only_when_you_were_pregnant?_If_Respondent_says_pre-diabetes_or_borderline_diabetes,_use_response_code_4.)
['Yes' 'No' 'Yes, but female told only during pregnancy' nan
 'No, pre-diabetes or borderline diabetes']


Column: Have_you_smoked_at_least_100_cigarettes_in_your_entire_life?_[Note:_5_packs_=_100_cigarettes]
['No' 'Yes' nan]


Column: Four-level_smoker_status:_Everyday_smoker,_Someday_smoker,_Former_smoker,_Non-smoker
['Never smoked' 'Former smoker' 'Current smoker' nan]


Column: Respondents_that_have_ever_reported_having_coronary_heart_disease_(CHD)_or_myocardial_infarction_(MI)
['Did not report having MI or CHD' 'Reported having MI or CHD' nan]


Column: (Ever_

### we will not delete all the rows with missing values we will just handel it diffrently 


In [11]:
# Drop duplicate rows based on all columns, keeping the first occurrence
df_selected.drop_duplicates(keep='first', inplace=True)


In [12]:
# missing percentage for each column
missing_percentage = df_selected.isnull().mean() * 100
print(missing_percentage)

(Ever_told)_(you_had)_a_stroke.                                                                                                                                                                 0.000000
Imputed_Age_value_collapsed_above_80                                                                                                                                                            0.000000
Sex_of_Respondent                                                                                                                                                                               0.000000
Body_Mass_Index_(BMI)                                                                                                                                                                           9.304072
Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional                                                                                        0.40

### how many rows have empty data actually in total 

In [13]:
# how many rows have empty data actually in total (that means all those rows which have more than one empty value in any column will be counted only once)
print("Total rows with empty data:", df_selected.isnull().any(axis=1).sum())

Total rows with empty data: 419393


In [14]:
df_selected.shape

(431553, 27)

In [15]:
# befor anything else I need to check the imbalance of the target variable (Ever_told_you_had_a_stroke) in the dataset. If the dataset is imbalanced, I will need to use techniques like oversampling, undersampling, or class weighting to address this issue.
df_selected[ '(Ever_told)_(you_had)_a_stroke.'].unique()

array(['No', 'Yes'], dtype=object)

### we cant delete any more columns because we will lose too much data, so we will have to impute the missing values instead. We will use the median for numerical columns and the mode for categorical columns.

In [16]:
# split the data into train and test sets 
target = "(Ever_told)_(you_had)_a_stroke."

X = df_selected.drop(columns=[target])

y = df_selected[target].map({
    "No": 0,
    "Yes": 1
})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [17]:
# missing percentage for each column
missing_percentage_train = X_train.isnull().mean() * 100
print(missing_percentage_train)


Imputed_Age_value_collapsed_above_80                                                                                                                                                            0.000000
Sex_of_Respondent                                                                                                                                                                               0.000000
Body_Mass_Index_(BMI)                                                                                                                                                                           9.305646
Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional                                                                                        0.409278
(Ever_told)_(you_had)_diabetes?_(If_Yes_-_and_respondent_is_female,_ask_Was_this_only_when_you_were_pregnant?_If_Respondent_says_pre-diabetes_or_borderline_diabetes,_use_response_code_4.)     0.16

In [18]:

missing_percentage_test = X_test.isnull().mean() * 100
print(missing_percentage_test)

Imputed_Age_value_collapsed_above_80                                                                                                                                                            0.000000
Sex_of_Respondent                                                                                                                                                                               0.000000
Body_Mass_Index_(BMI)                                                                                                                                                                           9.297772
Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional                                                                                        0.378862
(Ever_told)_(you_had)_diabetes?_(If_Yes_-_and_respondent_is_female,_ask_Was_this_only_when_you_were_pregnant?_If_Respondent_says_pre-diabetes_or_borderline_diabetes,_use_response_code_4.)     0.17

In [19]:
# for columns with a float values (Actual FLoat not some encoded float)
# but dont do it on real file before training testing split else data leakge will happen 
from sklearn.impute import SimpleImputer

numeric_cols = [
    '_AGE80',
    '_BMI5',
    'PAMIN13_',
    'PHYSHLTH',
    'MENTHLTH'
]

imputer = SimpleImputer(strategy='median')
col_name = []

for key, value in rename_map.items():
    if key in numeric_cols:
        col_name.append(value)

X_train[col_name] = imputer.fit_transform(
    X_train[col_name])
X_test[col_name] = imputer.transform(
    X_test[col_name])





In [20]:

missing_percentage_train = X_train.isnull().mean() * 100
print(missing_percentage_train)

Imputed_Age_value_collapsed_above_80                                                                                                                                                            0.000000
Sex_of_Respondent                                                                                                                                                                               0.000000
Body_Mass_Index_(BMI)                                                                                                                                                                           0.000000
Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional                                                                                        0.409278
(Ever_told)_(you_had)_diabetes?_(If_Yes_-_and_respondent_is_female,_ask_Was_this_only_when_you_were_pregnant?_If_Respondent_says_pre-diabetes_or_borderline_diabetes,_use_response_code_4.)     0.16

In [21]:
missing_percentage_test = X_test.isnull().mean() * 100
print(missing_percentage_test)

Imputed_Age_value_collapsed_above_80                                                                                                                                                            0.000000
Sex_of_Respondent                                                                                                                                                                               0.000000
Body_Mass_Index_(BMI)                                                                                                                                                                           0.000000
Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional                                                                                        0.378862
(Ever_told)_(you_had)_diabetes?_(If_Yes_-_and_respondent_is_female,_ask_Was_this_only_when_you_were_pregnant?_If_Respondent_says_pre-diabetes_or_borderline_diabetes,_use_response_code_4.)     0.17

In [22]:
# now categorical data imputation using mode
text_columns = X_train.select_dtypes(include="object").columns
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')
X_train[text_columns] = imputer.fit_transform(X_train[text_columns])
X_test[text_columns] = imputer.transform(X_test[text_columns])

In [23]:

missing_percentage_train = X_train.isnull().mean() * 100
print(missing_percentage_train)

Imputed_Age_value_collapsed_above_80                                                                                                                                                           0.0
Sex_of_Respondent                                                                                                                                                                              0.0
Body_Mass_Index_(BMI)                                                                                                                                                                          0.0
Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional                                                                                       0.0
(Ever_told)_(you_had)_diabetes?_(If_Yes_-_and_respondent_is_female,_ask_Was_this_only_when_you_were_pregnant?_If_Respondent_says_pre-diabetes_or_borderline_diabetes,_use_response_code_4.)    0.0
Have_you_smoked_at_least_

In [24]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 345242 entries, 125475 to 286639
Data columns (total 26 columns):
 #   Column                                                                                                                                                                                       Non-Null Count   Dtype  
---  ------                                                                                                                                                                                       --------------   -----  
 0   Imputed_Age_value_collapsed_above_80                                                                                                                                                         345242 non-null  float64
 1   Sex_of_Respondent                                                                                                                                                                            345242 non-null  object 
 2   Body_Mas

In [25]:
# change data type to float
X_train["Now_thinking_about_your_mental_health,_which_includes_stress,_depression,_and_problems_with_emotions,_for_how_many_days_during_the_past_30_days_was_your_mental_health_not_good?"] = X_train["Now_thinking_about_your_mental_health,_which_includes_stress,_depression,_and_problems_with_emotions,_for_how_many_days_during_the_past_30_days_was_your_mental_health_not_good?"].astype(float)
X_train["Now_thinking_about_your_physical_health,_which_includes_physical_illness_and_injury,_for_how_many_days_during_the_past_30_days_was_your_physical_health_not_good?"] = X_train["Now_thinking_about_your_physical_health,_which_includes_physical_illness_and_injury,_for_how_many_days_during_the_past_30_days_was_your_physical_health_not_good?"].astype(float)
X_test["Now_thinking_about_your_mental_health,_which_includes_stress,_depression,_and_problems_with_emotions,_for_how_many_days_during_the_past_30_days_was_your_mental_health_not_good?"] = X_test["Now_thinking_about_your_mental_health,_which_includes_stress,_depression,_and_problems_with_emotions,_for_how_many_days_during_the_past_30_days_was_your_mental_health_not_good?"].astype(float)
X_test["Now_thinking_about_your_physical_health,_which_includes_physical_illness_and_injury,_for_how_many_days_during_the_past_30_days_was_your_physical_health_not_good?"] = X_test["Now_thinking_about_your_physical_health,_which_includes_physical_illness_and_injury,_for_how_many_days_during_the_past_30_days_was_your_physical_health_not_good?"].astype(float)

#### save

In [26]:
"""# save train test data 
with open("../ dataset/processed/train1.csv", "w", encoding="utf-8") as f:
    train_data.to_csv(f, index=False)
with open("../ dataset/processed/test1.csv", "w", encoding="utf-8") as f:
    test_data.to_csv(f, index=False)
print("these data will be saved by pipline and also will be used for more frontend and user frendly interface use")"""

'# save train test data \nwith open("../ dataset/processed/train1.csv", "w", encoding="utf-8") as f:\n    train_data.to_csv(f, index=False)\nwith open("../ dataset/processed/test1.csv", "w", encoding="utf-8") as f:\n    test_data.to_csv(f, index=False)\nprint("these data will be saved by pipline and also will be used for more frontend and user frendly interface use")'

In [44]:
X_train.columns

Index(['Imputed_Age_value_collapsed_above_80', 'Sex_of_Respondent',
       'Body_Mass_Index_(BMI)',
       'Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional',
       '(Ever_told)_(you_had)_diabetes?_(If_Yes_-_and_respondent_is_female,_ask_Was_this_only_when_you_were_pregnant?_If_Respondent_says_pre-diabetes_or_borderline_diabetes,_use_response_code_4.)',
       'Have_you_smoked_at_least_100_cigarettes_in_your_entire_life?_[Note:_5_packs_=_100_cigarettes]',
       'Four-level_smoker_status:_Everyday_smoker,_Someday_smoker,_Former_smoker,_Non-smoker',
       'Respondents_that_have_ever_reported_having_coronary_heart_disease_(CHD)_or_myocardial_infarction_(MI)',
       '(Ever_told)_you_had_a_heart_attack,_also_called_a_myocardial_infarction?',
       '(Ever_told)_(you_had)_angina_or_coronary_heart_disease?',
       'Have_you_ever_been_told_by_a_doctor,_nurse_or_other_health_professional_that_your_cholesterol_is_high?',
       'Are_yo

Binary- Target (Ever_told)_(you_had)_a_stroke.
ordinal- general health, income, education
onehot - all else are here

In [27]:
# find all the column that have 2 unique values from object type columns
binary_cols = [col for col in X_train.columns if X_train[col].nunique() == 2 and X_train[col].dtype == 'object']
# find all the column that have 3 or more unique values 
multi_cols = [col for col in X_train.columns if X_train[col].nunique() >= 3 and X_train[col].dtype == 'object']
# find columns that have income, education or general health in the name
income_cols = [col for col in X_train.columns if 'income' in col.lower()]
education_cols = [col for col in X_train.columns if 'highest_grade' in col.lower()]
health_cols = [col for col in X_train.columns if 'general_your_health' in col.lower()]

In [28]:
print("Binary columns in train data:", binary_cols)
print("Multi-class columns in train data:", multi_cols)
print("Income columns:", income_cols)
print("Education columns:", education_cols)
print("Health columns:", health_cols)

Binary columns in train data: ['Sex_of_Respondent', 'Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional', 'Have_you_smoked_at_least_100_cigarettes_in_your_entire_life?_[Note:_5_packs_=_100_cigarettes]', 'Respondents_that_have_ever_reported_having_coronary_heart_disease_(CHD)_or_myocardial_infarction_(MI)', '(Ever_told)_you_had_a_heart_attack,_also_called_a_myocardial_infarction?', '(Ever_told)_(you_had)_angina_or_coronary_heart_disease?', 'Have_you_ever_been_told_by_a_doctor,_nurse_or_other_health_professional_that_your_cholesterol_is_high?', 'Are_you_currently_taking_medicine_prescribed_by_your_doctor_or_other_health_professional_for_your_cholesterol?', 'During_the_past_month,_other_than_your_regular_job,_did_you_participate_in_any_physical_activities_or_exercises_such_as_running,_calisthenics,_golf,_gardening,_or_walking_for_exercise?', 'Adults_who_reported_doing_physical_activity_or_exercise_during_the_past_30_days_other_than_the

In [29]:
# remove income, education, heath cols from multi class data
multi_cols.remove(income_cols[0])
multi_cols.remove(education_cols[0])
multi_cols.remove(health_cols[0])


In [30]:
ordinal = [health_cols[0],education_cols[0],income_cols[0]]
ordinal_categories = [

    # General Health
    [
        "Poor",
        "Fair",
        "Good",
        "Very good",
        "Excellent"
    ],

    # Education
    [
        "Never attended school or only kindergarten",
        "Grades 1 through 8 (Elementary)",
        "Grades 9 through 11 (Some high school)",
        "Grade 12 or GED (High school graduate)",
        "College 1 year to 3 years (Some college or technical school)",
        "College 4 years or more (College graduate)"
    ],

    # Annual Household Income
    [
        "Less than $10,000",
        "Less than $15,000 ($10,000 to < $15,000)",
        "Less than $20,000 ($15,000 to < $20,000)",
        "Less than $25,000 ($20,000 to < $25,000)",
        "Less than $35,000 ($25,000 to < $35,000)",
        "Less than $50,000 ($35,000 to < $50,000)",
        "Less than $75,000 ($50,000 to < $75,000)",
        "Less than $100,000 ($75,000 to < $100,000)",
        "Less than $150,000 ($100,000 to < $150,000)",
        "Less than $200,000 ($150,000 to < $200,000)",
        "$200,000 or more"
    ]
]
ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories
)

In [31]:
# column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("binary", OneHotEncoder(drop="if_binary"), binary_cols),
        ("ordinal", ordinal_encoder, ordinal),
        ("nominal", OneHotEncoder(handle_unknown="ignore"), multi_cols)
    ],
    remainder="passthrough"
)



In [40]:
# where df_selected index is 125475
df_selected.loc[125475]

(Ever_told)_(you_had)_a_stroke.                                                                                                                                                                                                               No
Imputed_Age_value_collapsed_above_80                                                                                                                                                                                                        36.0
Sex_of_Respondent                                                                                                                                                                                                                         Female
Body_Mass_Index_(BMI)                                                                                                                                                                                                                     2296.0
Adults_who_have_been_told_they_have_

- merge train data(X_train, y_train) and test data (X_test, y_test)

In [36]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [38]:
feature_names = preprocessor.get_feature_names_out()

X_train_transformed_df = pd.DataFrame(
    X_train_transformed.toarray() if hasattr(X_train_transformed, "toarray") else X_train_transformed,
    columns=feature_names,
    index=X_train.index
)
X_train_transformed_df


,binary__Sex_of_Respondent_Male,"binary__Adults_who_have_been_told_they_have_high_blood_pressure_by_a_doctor,_nurse,_or_other_health_professional_Yes",binary__Have_you_smoked_at_least_100_cigarettes_in_your_entire_life?_[Note:_5_packs_=_100_cigarettes]_Yes,binary__Respondents_that_have_ever_reported_having_coronary_heart_disease_(CHD)_or_myocardial_infarction_(MI)_Reported having MI or CHD,"binary__(Ever_told)_you_had_a_heart_attack,_also_called_a_myocardial_infarction?_Yes",binary__(Ever_told)_(you_had)_angina_or_coronary_heart_disease?_Yes,"binary__Have_you_ever_been_told_by_a_doctor,_nurse_or_other_health_professional_that_your_cholesterol_is_high?_Yes",binary__Are_you_currently_taking_medicine_prescribed_by_your_doctor_or_other_health_professional_for_your_cholesterol?_Yes,"binary__During_the_past_month,_other_than_your_regular_job,_did_you_participate_in_any_physical_activities_or_exercises_such_as_running,_calisthenics,_golf,_gardening,_or_walking_for_exercise?_Yes",binary__Adults_who_reported_doing_physical_activity_or_exercise_during_the_past_30_days_other_than_their_regular_job_No physical activity or exercise in last 30 days,...,nominal__Are_you:_(marital_status)_Divorced,nominal__Are_you:_(marital_status)_Married,nominal__Are_you:_(marital_status)_Never married,nominal__Are_you:_(marital_status)_Separated,nominal__Are_you:_(marital_status)_Widowed,remainder__Imputed_Age_value_collapsed_above_80,remainder__Body_Mass_Index_(BMI),remainder__Minutes_of_Physical_Activity_per_week_for_First_Activity,"remainder__Now_thinking_about_your_physical_health,_which_includes_physical_illness_and_injury,_for_how_many_days_during_the_past_30_days_was_your_physical_health_not_good?","remainder__Now_thinking_about_your_mental_health,_which_includes_stress,_depression,_and_problems_with_emotions,_for_how_many_days_during_the_past_30_days_was_your_mental_health_not_good?"
125475,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,36.0,2296.0,350.0,2.0,10.0
155743,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,59.0,2694.0,70.0,6.0,1.0
303651,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,63.0,3210.0,60.0,30.0,7.0
92951,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,46.0,3692.0,120.0,6.0,7.0
101191,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,42.0,2141.0,60.0,6.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120853,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,50.0,2585.0,180.0,6.0,7.0
110779,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,66.0,2829.0,150.0,6.0,7.0
217834,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,60.0,2870.0,720.0,6.0,7.0
428019,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,80.0,2421.0,90.0,6.0,7.0


In [41]:

X_test_transformed_df = pd.DataFrame(
    X_test_transformed.toarray() if hasattr(X_test_transformed, "toarray") else X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

In [42]:
X_train_transformed_df["remainder__Body_Mass_Index_(BMI)"] = X_train_transformed_df["remainder__Body_Mass_Index_(BMI)"]/100
X_test_transformed_df["remainder__Body_Mass_Index_(BMI)"] = X_test_transformed_df["remainder__Body_Mass_Index_(BMI)"]/100

In [43]:
# 
train_data = X_train_transformed_df.copy()
train_data["(Ever_told)_(you_had)_a_stroke."] = y_train
test_data = X_test_transformed_df.copy()
test_data["(Ever_told)_(you_had)_a_stroke."] = y_test

# Conclusion
We did the practice things now need to create a pipline for preprocessing but before that we should clearly writ what we will be doing in pipline to make it clean pipeline 

#### Steps
1. load data and interpretation. 
2. Convert SAS variable names into question-based column names and values in as as ber value label
3. Feature selection
4. interpret meaning of values and convert them in nan if they are not uninformative or remove the parts that are just clutter
5. Remove row that contains nan value for target variable for cleaning 
6. drop duplicates
7. Train test split to prevent data leak
8. impute numerical data through median
9. impute categorical data through mode 
10. select columns 
11. set the order
12. define ordinal encoder
13. define column transformer 
14. transform data 
15. store transformed data and data just before transformation